# FinTech Onboarding Funnel — Full EDA Dark Dashboard

Three-panel executive view: KPI cards, conversion funnel, and KYC loss by acquisition channel.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import psycopg2
import seaborn as sns
from matplotlib.gridspec import GridSpec

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "fintech_growth",
    "user": "admin",
    "password": "password123",
}

FUNNEL_QUERY = """
WITH funnel_stages AS (
    SELECT
        e.user_id,
        MAX(CASE WHEN e.event_name = 'account_created' THEN 1 ELSE 0 END) AS stage_1_signup,
        MAX(CASE WHEN e.event_name = 'document_uploaded' THEN 1 ELSE 0 END) AS stage_2_doc_upload,
        MAX(CASE WHEN e.event_name = 'kyc_approved' THEN 1 ELSE 0 END) AS stage_3_kyc_approved,
        MAX(CASE WHEN e.event_name = 'first_deposit_initiated' THEN 1 ELSE 0 END) AS stage_4_first_deposit
    FROM user_events e
    GROUP BY e.user_id
)
SELECT
    SUM(stage_1_signup) AS account_created,
    SUM(stage_2_doc_upload) AS document_uploaded,
    SUM(stage_3_kyc_approved) AS kyc_approved,
    SUM(stage_4_first_deposit) AS first_deposit
FROM funnel_stages;
"""

REVENUE_LEAKAGE_QUERY = """
WITH user_deposits AS (
    SELECT user_id, SUM(amount) AS total_deposited
    FROM financial_transactions
    WHERE transaction_type = 'deposit' AND status = 'completed'
    GROUP BY user_id
),
avg_revenue_per_converted_user AS (
    SELECT AVG(total_deposited) AS avg_deposit_val FROM user_deposits
),
lost_users AS (
    SELECT COUNT(DISTINCT user_id) AS lost_at_kyc_count
    FROM user_events
    WHERE user_id IN (
        SELECT user_id FROM user_events WHERE event_name = 'document_uploaded'
    )
      AND user_id NOT IN (
        SELECT user_id FROM user_events WHERE event_name = 'kyc_approved'
    )
)
SELECT
    lu.lost_at_kyc_count,
    ROUND(lu.lost_at_kyc_count * ar.avg_deposit_val, 2) AS estimated_lost_revenue
FROM lost_users lu
CROSS JOIN avg_revenue_per_converted_user ar;
"""

CHANNEL_KYC_LOSS_QUERY = """
WITH user_funnel AS (
    SELECT
        u.user_id,
        u.acquisition_channel,
        MAX(CASE WHEN e.event_name = 'document_uploaded' THEN 1 ELSE 0 END) AS uploaded_doc,
        MAX(CASE WHEN e.event_name = 'kyc_approved' THEN 1 ELSE 0 END) AS kyc_approved
    FROM users u
    LEFT JOIN user_events e ON e.user_id = u.user_id
    GROUP BY u.user_id, u.acquisition_channel
)
SELECT
    acquisition_channel,
    SUM(uploaded_doc) AS users_uploaded_doc,
    SUM(kyc_approved) AS users_kyc_approved,
    SUM(uploaded_doc) - SUM(kyc_approved) AS users_lost_at_kyc,
    ROUND(
        ((SUM(uploaded_doc) - SUM(kyc_approved))::NUMERIC / NULLIF(SUM(uploaded_doc), 0)) * 100,
        2
    ) AS kyc_dropoff_rate
FROM user_funnel
GROUP BY acquisition_channel
ORDER BY users_lost_at_kyc DESC;
"""

CHANNEL_DISPLAY = {
    "paid_social": "Paid Ads (Google/Meta)",
    "organic_search": "Organic Search",
    "referral": "Referral",
    "affiliate": "Affiliate",
    "app_store": "App Store",
    "email_campaign": "Email Campaign",
}

BG = "#0d1117"
PANEL = "#161b22"
CARD = "#21262d"
TEXT = "#e6edf3"
MUTED = "#8b949e"
BLUE = "#58a6ff"
BLUE_SOFT = "#388bfd"
RED = "#e63946"
AMBER = "#f4a261"

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
OUTPUT_PATH = PROJECT_ROOT / "dashboards" / "funnel_dashboard.png"

In [2]:
with psycopg2.connect(**DB_CONFIG) as conn:
    funnel_row = pd.read_sql(FUNNEL_QUERY, conn)
    leakage_row = pd.read_sql(REVENUE_LEAKAGE_QUERY, conn)
    channel_df = pd.read_sql(CHANNEL_KYC_LOSS_QUERY, conn)

counts = [
    int(funnel_row.loc[0, "account_created"]),
    int(funnel_row.loc[0, "document_uploaded"]),
    int(funnel_row.loc[0, "kyc_approved"]),
    int(funnel_row.loc[0, "first_deposit"]),
]
labels = [
    "Account Created",
    "Document Uploaded",
    "KYC Approved (Bottleneck)",
    "First Deposit",
]
step_rates = [
    100.0,
    round(counts[1] / counts[0] * 100, 1),
    round(counts[2] / counts[1] * 100, 1),
    round(counts[3] / counts[2] * 100, 1),
]

total_signups = counts[0]
kyc_dropoff_rate = round((1 - counts[2] / counts[1]) * 100, 2)
est_revenue_leakage = float(leakage_row.loc[0, "estimated_lost_revenue"])

channel_df["channel_label"] = channel_df["acquisition_channel"].map(
    lambda c: CHANNEL_DISPLAY.get(c, c.replace("_", " ").title())
)
channel_df

/var/folders/n7/vsn5r_fd4hgcc69fdgyvpsjw0000gn/T/tmp.MxaEYC6Hvv/ipykernel_33711/262323828.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  funnel_row = pd.read_sql(FUNNEL_QUERY, conn)
/var/folders/n7/vsn5r_fd4hgcc69fdgyvpsjw0000gn/T/tmp.MxaEYC6Hvv/ipykernel_33711/262323828.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  leakage_row = pd.read_sql(REVENUE_LEAKAGE_QUERY, conn)
/var/folders/n7/vsn5r_fd4hgcc69fdgyvpsjw0000gn/T/tmp.MxaEYC6Hvv/ipykernel_33711/262323828.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  cha

,acquisition_channel,users_uploaded_doc,users_kyc_approved,users_lost_at_kyc,kyc_dropoff_rate,channel_label
0,organic_search,230,132,98,42.61,Organic Search
1,paid_social,195,107,88,45.13,Paid Ads (Google/Meta)
2,referral,155,97,58,37.42,Referral
3,app_store,108,59,49,45.37,App Store
4,affiliate,95,56,39,41.05,Affiliate
5,email_campaign,57,29,28,49.12,Email Campaign


In [3]:
sns.set_theme(style="dark", context="talk")
plt.rcParams.update({
    "font.family": "sans-serif",
    "text.color": TEXT,
    "axes.labelcolor": MUTED,
    "xtick.color": MUTED,
    "ytick.color": TEXT,
})

fig = plt.figure(figsize=(15, 12), facecolor=BG)
gs = GridSpec(
    3, 3,
    figure=fig,
    height_ratios=[1.0, 2.4, 2.2],
    hspace=0.38,
    wspace=0.22,
    left=0.09,
    right=0.96,
    top=0.92,
    bottom=0.06,
)

# ---------------------------------------------------------------------------
# Panel 1 — KPI cards
# ---------------------------------------------------------------------------
kpi_specs = [
    {
        "title": "Total Users",
        "value": f"{total_signups:,}",
        "accent": BLUE,
        "subtitle": "6-month cohort",
    },
    {
        "title": "KYC Drop-off Rate",
        "value": f"{kyc_dropoff_rate:.2f}%",
        "accent": RED,
        "subtitle": "Primary bottleneck",
    },
    {
        "title": "Estimated Lost Volume",
        "value": f"${est_revenue_leakage/1000:.0f}k",
        "accent": AMBER,
        "subtitle": "Initial deposit leakage",
    },
]

for i, kpi in enumerate(kpi_specs):
    ax = fig.add_subplot(gs[0, i])
    ax.set_facecolor(CARD)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.add_patch(
        mpatches.FancyBboxPatch(
            (0.02, 0.12), 0.015, 0.76,
            boxstyle="round,pad=0.01,rounding_size=0.01",
            linewidth=0,
            facecolor=kpi["accent"],
            transform=ax.transAxes,
            clip_on=False,
        )
    )
    ax.text(0.10, 0.72, kpi["title"], transform=ax.transAxes,
            fontsize=11, color=MUTED, ha="left", va="center")
    ax.text(0.10, 0.42, kpi["value"], transform=ax.transAxes,
            fontsize=26, color=kpi["accent"], fontweight="bold", ha="left", va="center")
    ax.text(0.10, 0.18, kpi["subtitle"], transform=ax.transAxes,
            fontsize=9, color=MUTED, ha="left", va="center")

# ---------------------------------------------------------------------------
# Panel 2 — Funnel chart
# ---------------------------------------------------------------------------
ax_funnel = fig.add_subplot(gs[1, :])
ax_funnel.set_facecolor(PANEL)

bar_colors = [BLUE_SOFT, BLUE, RED, BLUE_SOFT]
y_pos = list(range(len(labels)))
bars = ax_funnel.barh(
    y_pos, counts, color=bar_colors, height=0.62,
    edgecolor=PANEL, linewidth=1.5, zorder=3,
)

ax_funnel.set_yticks(y_pos)
ax_funnel.set_yticklabels(labels, fontsize=12)
ax_funnel.invert_yaxis()
ax_funnel.set_xlabel("Users", fontsize=11, color=MUTED)
ax_funnel.set_xlim(0, max(counts) * 1.30)
ax_funnel.grid(axis="x", color="#30363d", linestyle="--", linewidth=0.8, zorder=0)
ax_funnel.set_axisbelow(True)
ax_funnel.set_title("Conversion Funnel", loc="left", fontsize=13, color=TEXT, pad=10)

for spine in ax_funnel.spines.values():
    spine.set_color("#30363d")
ax_funnel.tick_params(colors=MUTED)

for bar, count, rate in zip(bars, counts, step_rates):
    ax_funnel.text(
        bar.get_width() + max(counts) * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{count:,}  ({rate:.1f}% conversion)",
        va="center", ha="left", fontsize=11, fontweight="bold", color=TEXT, zorder=4,
    )

# ---------------------------------------------------------------------------
# Panel 3 — KYC loss by acquisition channel
# ---------------------------------------------------------------------------
ax_ch = fig.add_subplot(gs[2, :])
ax_ch.set_facecolor(PANEL)

channel_plot = channel_df.sort_values("users_lost_at_kyc", ascending=True)
ch_colors = [RED if c == "paid_social" else BLUE_SOFT for c in channel_plot["acquisition_channel"]]

bars_ch = ax_ch.barh(
    channel_plot["channel_label"],
    channel_plot["users_lost_at_kyc"],
    color=ch_colors,
    height=0.62,
    edgecolor=PANEL,
    linewidth=1.2,
    zorder=3,
)

ax_ch.set_xlabel("Users Lost at KYC", fontsize=11, color=MUTED)
ax_ch.set_title(
    "KYC Drop-off by Acquisition Channel (Paid Ads vs Organic vs Referral)",
    loc="left", fontsize=13, color=TEXT, pad=10,
)
ax_ch.grid(axis="x", color="#30363d", linestyle="--", linewidth=0.8, zorder=0)
ax_ch.set_axisbelow(True)
for spine in ax_ch.spines.values():
    spine.set_color("#30363d")
ax_ch.tick_params(colors=MUTED)

xmax = max(channel_plot["users_lost_at_kyc"].max() * 1.25, 1)
ax_ch.set_xlim(0, xmax)

for bar, (_, row) in zip(bars_ch, channel_plot.iterrows()):
    ax_ch.text(
        bar.get_width() + xmax * 0.015,
        bar.get_y() + bar.get_height() / 2,
        f"{int(row['users_lost_at_kyc']):,} lost  ({row['kyc_dropoff_rate']:.1f}% drop-off)",
        va="center", ha="left", fontsize=10, fontweight="bold", color=TEXT, zorder=4,
    )

fig.suptitle(
    "Neobank Onboarding Funnel — Executive Conversion Dashboard",
    fontsize=16, fontweight="bold", color=TEXT, y=0.97,
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(OUTPUT_PATH, dpi=300, bbox_inches="tight", facecolor=BG, edgecolor="none")
plt.show()

print(f"Dashboard saved to: {OUTPUT_PATH}")
print(
    f"KPIs -> Users: {total_signups:,} | KYC Drop-off: {kyc_dropoff_rate:.2f}% | "
    f"Lost Volume: ${est_revenue_leakage:,.0f}"
)

Dashboard saved to: /Users/caue/Data Analyst portifolio/FinTech Growth & Revenue Funnel/dashboards/funnel_dashboard.png
KPIs -> Users: 1,200 | KYC Drop-off: 42.86% | Lost Volume: $393,175


/var/folders/n7/vsn5r_fd4hgcc69fdgyvpsjw0000gn/T/tmp.MxaEYC6Hvv/ipykernel_33711/1904661929.py:156: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
